In [2]:
# CELL 0: ENVIRONMENT DIAGNOSTICS
import os, sys, subprocess

print("=" * 70)
print("🖥️  HARDWARE")
print("=" * 70)

# GPU
print("\nGPU:")
os.system("nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv,noheader")

# CUDA + PyTorch
print(f"\nCUDA available: ", end="")
try:
    import torch
    print(f"Yes — torch {torch.__version__}, CUDA {torch.version.cuda}")
    print(f"GPU via torch: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
except:
    print("No torch found yet")

print(f"\nPython: {sys.version}")

print("\n" + "=" * 70)
print("💾  DISK SPACE")
print("=" * 70)

print("\nContainer (/):")
os.system("df -h / | tail -1")
print("\nWorkspace volume (/workspace):")
os.system("df -h /workspace | tail -1")

print("\n" + "=" * 70)
print("📂  WORKSPACE CONTENTS")
print("=" * 70)

if os.path.exists("/workspace"):
    contents = os.listdir("/workspace")
    print(f"\n/workspace/ has {len(contents)} items:")
    for item in sorted(contents):
        path = f"/workspace/{item}"
        if os.path.isdir(path):
            print(f"  📁 {item}/")
        else:
            size = os.path.getsize(path)
            print(f"  📄 {item} ({size/1024:.1f} KB)")
else:
    print("\n⚠️  /workspace does not exist!")

print("\n" + "=" * 70)
print("📦  PACKAGE STATUS")
print("=" * 70)

packages = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "accelerate": "accelerate",
    "numpy": "numpy",
    "scipy": "scipy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "huggingface_hub": "huggingface_hub",
    "einops": "einops",
    "bitsandbytes": "bitsandbytes",
}

installed = []
missing = []
for display, imp in packages.items():
    try:
        mod = __import__(imp)
        ver = getattr(mod, '__version__', 'ok')
        installed.append(f"  ✅ {display}: {ver}")
    except ImportError:
        missing.append(f"  ❌ {display}")

print("\nInstalled:")
for p in installed:
    print(p)
if missing:
    print("\nMissing:")
    for p in missing:
        print(p)



🖥️  HARDWARE

GPU:
NVIDIA A100-SXM4-80GB, 81920 MiB, 81149 MiB, 580.126.16

CUDA available: Yes — torch 2.4.1+cu124, CUDA 12.4
GPU via torch: NVIDIA A100-SXM4-80GB
No torch found yet

Python: 3.11.10 (main, Sep  7 2024, 18:35:41) [GCC 11.4.0]

💾  DISK SPACE

Container (/):
overlay          20G   16M   20G   1% /

Workspace volume (/workspace):
mfs#us-md-1.runpod.net:9421  299T  226T   74T  76% /workspace

📂  WORKSPACE CONTENTS

/workspace/ has 2 items:
  📁 .ipynb_checkpoints/
  📄 circuittier.ipynb (0.6 KB)

📦  PACKAGE STATUS

Installed:
  ✅ torch: 2.4.1+cu124
  ✅ numpy: 1.26.3

Missing:
  ❌ transformers
  ❌ datasets
  ❌ accelerate
  ❌ scipy
  ❌ pandas
  ❌ matplotlib
  ❌ tqdm
  ❌ huggingface_hub
  ❌ einops
  ❌ bitsandbytes


In [3]:
# CELL 1: INSTALL PACKAGES
import time
start = time.time()

print("📦 Installing dependencies...")
print("=" * 60)

# Core ML
!pip install -q transformers accelerate

# HuggingFace
!pip install -q huggingface_hub datasets

# Math & Viz
!pip install -q scipy pandas matplotlib seaborn

# Utils
!pip install -q tqdm einops

# For 8-bit Adam optimizer (saves VRAM during finetuning)
!pip install -q bitsandbytes

# Remove torchvision/torchaudio (conflict prevention)
!pip uninstall -y torchvision torchaudio -q 2>/dev/null

elapsed = time.time() - start
print(f"\n✅ Done in {elapsed:.0f}s")

# Verify critical imports
print("\n" + "=" * 60)
print("🔍 Verifying...")
print("=" * 60)

import torch, transformers, accelerate, numpy, scipy
from huggingface_hub import login
from tqdm.auto import tqdm

print(f"  torch:        {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  transformers: {transformers.__version__}")
print(f"  accelerate:   {accelerate.__version__}")
print(f"  numpy:        {numpy.__version__}")
print(f"  scipy:        {scipy.__version__}")
print(f"  huggingface_hub: ✅")
print(f"  bitsandbytes: ✅")

print("\n✅ All packages ready")

📦 Installing dependencies...

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

✅ Done in 31s

🔍 Verifying...


AttributeError: module 'numpy._core' has no attribute 'multiarray'

In [4]:
# CELL 2: FIX NUMPY + ACCELERATE CONFLICT
!pip install -q "numpy>=1.26.4" --upgrade
!pip install -q "accelerate==1.2.1"

print("✅ Fixed. Restarting kernel...")
print("⚠️  Go to Kernel → Restart Kernel, then run Cell 3")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
✅ Fixed. Restarting kernel...
⚠️  Go to Kernel → Restart Kernel, then run Cell 3


In [2]:
# CELL 3: VERIFY ALL IMPORTS (post-restart)

import torch
import transformers
import accelerate
import numpy as np
import scipy
from huggingface_hub import login
from tqdm.auto import tqdm

print(f"  torch:        {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  GPU:          {torch.cuda.get_device_name(0)}")
print(f"  VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  transformers: {transformers.__version__}")
print(f"  accelerate:   {accelerate.__version__}")
print(f"  numpy:        {np.__version__}")
print(f"  scipy:        {scipy.__version__}")

import datasets, pandas, matplotlib, bitsandbytes
print(f"  datasets:     {datasets.__version__}")
print(f"  bitsandbytes: {bitsandbytes.__version__}")
print(f"  pandas/matplotlib/tqdm: ✅")

print(f"\n✅ All good. Ready for HuggingFace login.")

  torch:        2.4.1+cu124 (CUDA: True)
  GPU:          NVIDIA A100-SXM4-80GB
  VRAM:         85.1 GB
  transformers: 5.4.0
  accelerate:   1.2.1
  numpy:        2.4.4
  scipy:        1.17.1
  datasets:     4.8.4
  bitsandbytes: 0.49.2
  pandas/matplotlib/tqdm: ✅

✅ All good. Ready for HuggingFace login.


In [3]:
# CELL 4: HUGGINGFACE LOGIN
from huggingface_hub import login

login()
print("✅ Logged in")

✅ Logged in


In [4]:
# CELL 5: DOWNLOAD + INSPECT DATASET
from datasets import load_dataset

print("📥 Downloading neo4j/text2cypher-2024v1...")
ds = load_dataset("neo4j/text2cypher-2024v1")

print(f"   Train: {len(ds['train']):,} samples")
print(f"   Test:  {len(ds['test']):,} samples")
print(f"   Columns: {ds['train'].column_names}")

# Check data sources
from collections import Counter
sources = Counter(ds['train']['data_source'])
print(f"\n📊 Data sources ({len(sources)} unique):")
for src, count in sources.most_common():
    print(f"   {src:<45} {count:>6}")

# Check schema lengths
schema_lens = [len(s) for s in ds['train']['schema']]
print(f"\n📏 Schema lengths (chars):")
print(f"   Min: {min(schema_lens)}, Max: {max(schema_lens)}, Mean: {sum(schema_lens)//len(schema_lens)}")

# Buckets
for threshold in [200, 500, 1000, 2000, 4000]:
    count = sum(1 for l in schema_lens if l <= threshold)
    print(f"   ≤{threshold:>5}: {count:>6} ({100*count/len(schema_lens):.1f}%)")

# Show 3 samples
print(f"\n📝 Sample entries:")
for i in [0, 100, 1000]:
    s = ds['train'][i]
    print(f"\n--- Sample {i} ---")
    print(f"   Source:  {s['data_source']}")
    print(f"   Question: {s['question'][:80]}")
    print(f"   Schema:  {len(s['schema'])} chars")
    print(f"   Cypher:  {s['cypher'][:120]}")

📥 Downloading neo4j/text2cypher-2024v1...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/7.33M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/835k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39554 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4833 [00:00<?, ? examples/s]

   Train: 39,554 samples
   Test:  4,833 samples
   Columns: ['question', 'schema', 'cypher', 'data_source', 'instance_id', 'database_reference_alias']

📊 Data sources (20 unique):
   neo4jLabs_functional_cypher                    13571
   neo4jLabs_synthetic_gpt4turbo                   6348
   neo4jLabs_synthetic_gpt4o                       6106
   neo4jLabs_synthetic_gemini                      5895
   neo4jLabs_synthetic_claudeopus                  3257
   neo4j_text2cypher2023_train                     2587
   neo4j_crowdsourced                               487
   cyspider_t5base_prefix_correct                   324
   hf_vedana17_train                                228
   neo4j_rageval_products                           203
   hf_iprahara                                      136
   hf_dfwlab_train                                  119
   neo4j_rageval_movies                              64
   cyspider_cased_train                              53
   cyspider_t5base_incorrect       

In [5]:
# CELL 6: PREPROCESS — SCHEMA TRUNCATION + PROMPT FORMATTING + COMPLEXITY TAGGING

import re, json, os
from collections import Counter
from transformers import AutoTokenizer

print("📦 Loading Llama-3 tokenizer (for token counting)...")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")

# ============================================================
# SCHEMA TRUNCATION
# ============================================================

def truncate_schema(schema, max_chars=800):
    """Truncate long schemas: keep node labels, property names, relationships."""
    if len(schema) <= max_chars:
        return schema
    
    # JSON format
    if schema.strip().startswith('{'):
        return _truncate_json_schema(schema, max_chars)
    
    # Verbose "Node properties" format
    if 'Node properties' in schema or '**' in schema:
        return _truncate_verbose_schema(schema, max_chars)
    
    # Compact "Graph schema" or "Relevant node labels" format
    if 'Relevant node labels' in schema or 'Graph schema' in schema:
        return schema[:max_chars]
    
    return schema[:max_chars]


def _truncate_verbose_schema(schema, max_chars):
    nodes = []
    relationships_lines = []
    rel_prop_names = []
    
    current_node = None
    current_props = []
    in_relationships = False
    in_rel_props = False
    
    for line in schema.split('\n'):
        ls = line.strip()
        
        if ls.startswith('The relationships:') or (ls.startswith('(:') and not in_rel_props):
            in_relationships = True
            in_rel_props = False
            if current_node:
                nodes.append((current_node, current_props))
                current_node = None
                current_props = []
            if ls.startswith('(:'):
                relationships_lines.append(ls)
            continue
        
        if ls.startswith('Relationship properties:'):
            if current_node:
                nodes.append((current_node, current_props))
                current_node = None
                current_props = []
            in_rel_props = True
            in_relationships = False
            continue
        
        if in_rel_props:
            m = re.search(r'\*\*(.*?)\*\*', ls)
            if m:
                rel_prop_names.append(m.group(1))
            continue
        
        if in_relationships:
            if ls.startswith('(:') or ls.startswith('('):
                relationships_lines.append(ls)
            continue
        
        # Node definition
        if ls.startswith('- **') and '`' not in ls:
            if current_node:
                nodes.append((current_node, current_props))
            m = re.search(r'\*\*(.*?)\*\*', ls)
            current_node = m.group(1) if m else ls
            current_props = []
            continue
        
        # Also catch "- **Node** \n  - `prop`" pattern where node + first prop on same line
        if ls.startswith('- **') and '`' in ls:
            if current_node:
                nodes.append((current_node, current_props))
            m = re.search(r'\*\*(.*?)\*\*', ls)
            current_node = m.group(1) if m else ls
            current_props = []
            pm = re.search(r'`(\w+)`', ls)
            if pm:
                current_props.append(pm.group(1))
            continue
        
        # Property line
        if current_node and '`' in ls:
            pm = re.search(r'`(\w+)`', ls)
            if pm:
                current_props.append(pm.group(1))
    
    if current_node:
        nodes.append((current_node, current_props))
    
    # Build compact output
    parts = []
    if nodes:
        parts.append("Nodes:")
        for name, props in nodes:
            if props:
                short = props[:8]
                extra = f", +{len(props)-8} more" if len(props) > 8 else ""
                parts.append(f"  {name} ({', '.join(short)}{extra})")
            else:
                parts.append(f"  {name}")
    
    if rel_prop_names:
        parts.append(f"Relationship properties: {', '.join(rel_prop_names)}")
    
    if relationships_lines:
        parts.append("Relationships:")
        for r in relationships_lines:
            parts.append(f"  {r}")
    
    result = '\n'.join(parts)
    return result[:max_chars] if len(result) > max_chars else result


def _truncate_json_schema(schema, max_chars):
    try:
        data = json.loads(schema)
    except json.JSONDecodeError:
        return schema[:max_chars]
    
    parts = ["Nodes:"]
    rels = []
    
    for key, value in data.items():
        if isinstance(value, dict):
            if value.get('type') == 'node':
                props = list(value.get('properties', {}).keys())[:8]
                parts.append(f"  {key} ({', '.join(props)})" if props else f"  {key}")
                for rn, ri in value.get('relationships', {}).items():
                    direction = ri.get('direction', '')
                    for lbl in ri.get('labels', []):
                        if direction == 'out':
                            rels.append(f"(:{key})-[:{rn}]->(:{lbl})")
                        else:
                            rels.append(f"(:{lbl})-[:{rn}]->(:{key})")
    
    if rels:
        parts.append("Relationships:")
        for r in rels:
            parts.append(f"  {r}")
    
    result = '\n'.join(parts)
    return result[:max_chars] if len(result) > max_chars else result


# ============================================================
# COMPLEXITY TAGGING
# ============================================================

def tag_complexity(cypher):
    c = cypher.upper()
    has_where = "WHERE" in c
    has_rel = bool(re.search(r'\[.*?\]', cypher)) or "->" in cypher or "<-" in cypher
    has_agg = any(kw in c for kw in ["COUNT(", "SUM(", "AVG(", "MAX(", "MIN("])
    has_orderby = "ORDER BY" in c
    has_varpath = bool(re.search(r'\[\*', cypher))
    has_shortest = "SHORTESTPATH" in c
    has_multi_match = c.count("MATCH") > 1
    has_exists = "EXISTS" in c
    has_unwind = "UNWIND" in c
    has_case = "CASE " in c
    has_optional = "OPTIONAL" in c
    
    if has_shortest or has_varpath or has_multi_match or has_exists or has_unwind or has_case or has_optional:
        return "CY5"
    if has_agg and has_rel:
        return "CY4"
    if has_rel and (has_where or has_orderby):
        return "CY3"
    if has_rel or has_where or has_orderby:
        return "CY2"
    return "CY1"


# ============================================================
# PROCESS ALL DATA
# ============================================================

print("\n🔧 Processing dataset...")

def process_split(split_data, split_name):
    processed = []
    orig_lens = []
    trunc_lens = []
    
    for i in range(len(split_data)):
        row = split_data[i]
        
        schema_orig = row['schema']
        schema_trunc = truncate_schema(schema_orig)
        
        prompt = f"Schema: {schema_trunc}\nQuestion: {row['question']}\nCypher:"
        completion = " " + row['cypher']
        full_text = prompt + completion
        
        tokens = tokenizer(full_text, truncation=False)['input_ids']
        complexity = tag_complexity(row['cypher'])
        
        orig_lens.append(len(schema_orig))
        trunc_lens.append(len(schema_trunc))
        
        processed.append({
            'prompt': prompt,
            'completion': completion,
            'full_text': full_text,
            'cypher_ref': row['cypher'],
            'question': row['question'],
            'data_source': row['data_source'],
            'complexity': complexity,
            'n_tokens': len(tokens),
            'schema_orig_len': len(schema_orig),
            'schema_trunc_len': len(schema_trunc),
            'instance_id': row['instance_id'],
        })
    
    print(f"\n   {split_name}: {len(processed)} samples")
    print(f"   Schema chars — before: mean={sum(orig_lens)//len(orig_lens)}, max={max(orig_lens)}")
    print(f"   Schema chars — after:  mean={sum(trunc_lens)//len(trunc_lens)}, max={max(trunc_lens)}")
    
    token_lens = [p['n_tokens'] for p in processed]
    print(f"   Token lengths — mean={sum(token_lens)//len(token_lens)}, max={max(token_lens)}")
    for t in [256, 512, 768, 1024]:
        count = sum(1 for l in token_lens if l <= t)
        print(f"     ≤{t}: {count} ({100*count/len(token_lens):.1f}%)")
    
    complexity_counts = Counter(p['complexity'] for p in processed)
    print(f"   Complexity: {dict(sorted(complexity_counts.items()))}")
    
    return processed

train_processed = process_split(ds['train'], "Train")
test_processed = process_split(ds['test'], "Test")

# ============================================================
# SAVE TO WORKSPACE
# ============================================================

os.makedirs("/workspace/data", exist_ok=True)

# Save full processed data
with open("/workspace/data/train_processed.json", "w") as f:
    json.dump(train_processed, f)
print(f"\n💾 Saved train: {len(train_processed)} samples")

with open("/workspace/data/test_processed.json", "w") as f:
    json.dump(test_processed, f)
print(f"💾 Saved test: {len(test_processed)} samples")

# Create analysis split (500 from train, for signal extraction later)
# Pick evenly across complexity levels
import random
random.seed(42)

by_complexity = {}
for item in train_processed:
    by_complexity.setdefault(item['complexity'], []).append(item)

analysis_samples = []
for cy in sorted(by_complexity.keys()):
    pool = by_complexity[cy]
    n_pick = min(100, len(pool))
    analysis_samples.extend(random.sample(pool, n_pick))

random.shuffle(analysis_samples)
analysis_samples = analysis_samples[:500]

with open("/workspace/data/analysis_500.json", "w") as f:
    json.dump(analysis_samples, f)
print(f"💾 Saved analysis: {len(analysis_samples)} samples (for signal extraction)")

# Save per-complexity test splits
for cy in sorted(set(p['complexity'] for p in test_processed)):
    cy_data = [p for p in test_processed if p['complexity'] == cy]
    with open(f"/workspace/data/test_{cy}.json", "w") as f:
        json.dump(cy_data, f)
    print(f"   test_{cy}: {len(cy_data)} samples")

print(f"\n✅ All data saved to /workspace/data/")

📦 Loading Llama-3 tokenizer (for token counting)...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]


🔧 Processing dataset...

   Train: 39554 samples
   Schema chars — before: mean=1913, max=10438
   Schema chars — after:  mean=428, max=800
   Token lengths — mean=196, max=1027
     ≤256: 27006 (68.3%)
     ≤512: 39370 (99.5%)
     ≤768: 39550 (100.0%)
     ≤1024: 39553 (100.0%)
   Complexity: {'CY1': 1972, 'CY2': 12984, 'CY3': 9196, 'CY4': 7977, 'CY5': 7425}

   Test: 4833 samples
   Schema chars — before: mean=2025, max=10863
   Schema chars — after:  mean=443, max=800
   Token lengths — mean=196, max=719
     ≤256: 3282 (67.9%)
     ≤512: 4827 (99.9%)
     ≤768: 4833 (100.0%)
     ≤1024: 4833 (100.0%)
   Complexity: {'CY1': 303, 'CY2': 1700, 'CY3': 1084, 'CY4': 915, 'CY5': 831}

💾 Saved train: 39554 samples
💾 Saved test: 4833 samples
💾 Saved analysis: 500 samples (for signal extraction)
   test_CY1: 303 samples
   test_CY2: 1700 samples
   test_CY3: 1084 samples
   test_CY4: 915 samples
   test_CY5: 831 samples

✅ All data saved to /workspace/data/


In [6]:
# CELL 7: DOWNLOAD LLAMA-3-8B BASE MODEL
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["HF_HOME"] = "/workspace/.cache/huggingface"
os.makedirs("/workspace/models/base", exist_ok=True)

MODEL_ID = "meta-llama/Meta-Llama-3-8B"

print(f"📥 Downloading {MODEL_ID}...")
print("   This will take 5-10 minutes (~16GB)...")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"\n✅ Model loaded")
print(f"   Parameters:  {model.num_parameters():,}")
print(f"   Device:      {next(model.parameters()).device}")
print(f"   VRAM used:   {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"   Layers: {model.config.num_hidden_layers}")
print(f"   Heads:  {model.config.num_attention_heads} (KV: {model.config.num_key_value_heads})")
print(f"   Hidden: {model.config.hidden_size}")
print(f"   MLP:    {model.config.intermediate_size}")

# Quick test — feed it a Cypher prompt, confirm it gets 0%
print("\n" + "=" * 60)
print("🧪 Base model Cypher test (should produce garbage):")
print("=" * 60)

test_prompt = "Schema: Nodes:\n  Movie (title, year)\n  Actor (name)\nRelationships:\n  (:Actor)-[:ACTED_IN]->(:Movie)\nQuestion: Find all movies from 2020\nCypher:"

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.pad_token_id)
gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print(f"   Generated: {gen[:200]}")
print(f"   {'❌ Not Cypher (expected!)' if not gen.strip().upper().startswith('MATCH') else '⚠️ Looks like Cypher — base model might already know some'}")

print(f"\n🎯 Ready for finetuning")

📥 Downloading meta-llama/Meta-Llama-3-8B...
   This will take 5-10 minutes (~16GB)...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=80) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ Model loaded
   Parameters:  8,030,261,248
   Device:      cuda:0
   VRAM used:   16.06 GB
   Layers: 32
   Heads:  32 (KV: 8)
   Hidden: 4096
   MLP:    14336

🧪 Base model Cypher test (should produce garbage):
   Generated: MATCH (m:Movie)-[:ACTED_IN]-(a:Actor) WHERE m.year = 2020 RETURN m.title AS title, a.name AS actor
Explanation: This query finds all movies from 2020 by traversing from the Movie node to the Actor nod
   ⚠️ Looks like Cypher — base model might already know some

🎯 Ready for finetuning


In [7]:
# CELL 8: BASE MODEL ACCURACY ON ACTUAL TEST SET
# How much Cypher does Llama-3-8B already know?

import json, torch
from tqdm.auto import tqdm

print("🧪 TESTING BASE MODEL ON ACTUAL TEXT2CYPHER TEST SET")
print("   If accuracy is >20%, we have a problem.")
print("   If accuracy is <5%, we're fine.")
print("=" * 60)

with open("/workspace/data/test_processed.json") as f:
    test_data = json.load(f)

# Test on 200 samples (40 per complexity level)
import random
random.seed(42)

test_subset = []
by_cy = {}
for item in test_data:
    by_cy.setdefault(item['complexity'], []).append(item)

for cy in sorted(by_cy.keys()):
    pool = by_cy[cy]
    test_subset.extend(random.sample(pool, min(40, len(pool))))

random.shuffle(test_subset)
print(f"   Testing on {len(test_subset)} samples across CY1-CY5")

# Evaluate
exact = 0
valid_cypher = 0
per_cy = {}

for item in tqdm(test_subset, desc="Evaluating"):
    inputs = tokenizer(item['prompt'], return_tensors="pt", truncation=True, max_length=480).to(model.device)
    
    with torch.no_grad():
        out = model.generate(
            **inputs, 
            max_new_tokens=150, 
            do_sample=False, 
            pad_token_id=tokenizer.pad_token_id
        )
    
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    first_line = gen.split('\n')[0].strip()
    
    expected = item['cypher_ref'].strip()
    
    is_exact = (first_line == expected)
    is_cypher = first_line.upper().startswith('MATCH') or first_line.upper().startswith('RETURN') or first_line.upper().startswith('CALL')
    
    if is_exact:
        exact += 1
    if is_cypher:
        valid_cypher += 1
    
    cy = item['complexity']
    if cy not in per_cy:
        per_cy[cy] = {'exact': 0, 'valid': 0, 'total': 0}
    per_cy[cy]['total'] += 1
    if is_exact:
        per_cy[cy]['exact'] += 1
    if is_cypher:
        per_cy[cy]['valid'] += 1

print(f"\n{'='*60}")
print(f"📊 BASE MODEL RESULTS")
print(f"{'='*60}")
print(f"   Exact match: {exact}/{len(test_subset)} ({100*exact/len(test_subset):.1f}%)")
print(f"   Valid Cypher: {valid_cypher}/{len(test_subset)} ({100*valid_cypher/len(test_subset):.1f}%)")

print(f"\n   Per complexity:")
for cy in sorted(per_cy.keys()):
    d = per_cy[cy]
    print(f"   {cy}: exact={d['exact']}/{d['total']} ({100*d['exact']/d['total']:.1f}%), valid={d['valid']}/{d['total']} ({100*d['valid']/d['total']:.1f}%)")

pct = 100 * exact / len(test_subset)
print(f"\n{'='*60}")
if pct < 5:
    print(f"✅ Base model gets {pct:.1f}% — clean delta, proceed with finetuning")
elif pct < 20:
    print(f"⚠️  Base model gets {pct:.1f}% — noisy but workable, signals will still be detectable")
else:
    print(f"❌ Base model gets {pct:.1f}% — too much pre-existing knowledge, consider different task/model")
print(f"{'='*60}")

# Save results
with open("/workspace/data/base_model_accuracy.json", "w") as f:
    json.dump({
        'exact_match': exact, 'valid_cypher': valid_cypher,
        'total': len(test_subset), 'per_complexity': per_cy
    }, f, indent=2)

🧪 TESTING BASE MODEL ON ACTUAL TEXT2CYPHER TEST SET
   If accuracy is >20%, we have a problem.
   If accuracy is <5%, we're fine.
   Testing on 200 samples across CY1-CY5


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


📊 BASE MODEL RESULTS
   Exact match: 0/200 (0.0%)
   Valid Cypher: 194/200 (97.0%)

   Per complexity:
   CY1: exact=0/40 (0.0%), valid=39/40 (97.5%)
   CY2: exact=0/40 (0.0%), valid=39/40 (97.5%)
   CY3: exact=0/40 (0.0%), valid=39/40 (97.5%)
   CY4: exact=0/40 (0.0%), valid=39/40 (97.5%)
   CY5: exact=0/40 (0.0%), valid=38/40 (95.0%)

✅ Base model gets 0.0% — clean delta, proceed with finetuning


In [8]:
# CELL 9: TOKENIZER AUDIT
print("🔍 TOKENIZER CHECK")
print("=" * 60)

print(f"  Vocab size:      {tokenizer.vocab_size}")
print(f"  EOS token:       '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"  BOS token:       '{tokenizer.bos_token}' (id={tokenizer.bos_token_id})")
print(f"  PAD token:       '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"  PAD == EOS:      {tokenizer.pad_token_id == tokenizer.eos_token_id}")

# Check special tokens
print(f"\n  All special tokens: {tokenizer.all_special_tokens}")
print(f"  All special IDs:    {tokenizer.all_special_ids}")

# Test tokenization of a sample prompt+completion
sample_text = "Schema: Nodes:\n  Movie (title)\nQuestion: Find movies\nCypher: MATCH (m:Movie) RETURN m.title"
tokens = tokenizer(sample_text, return_tensors="pt")
decoded = tokenizer.decode(tokens['input_ids'][0])

print(f"\n  Sample text length: {len(sample_text)} chars")
print(f"  Token count:        {tokens['input_ids'].shape[1]}")
print(f"  Round-trip match:   {decoded.strip() == sample_text.strip()}")

# Check if there's an unused token we can use as pad instead
print(f"\n  Token 128001: '{tokenizer.decode([128001])}'")
print(f"  Token 128002: '{tokenizer.decode([128002])}'")
print(f"  Token 128003: '{tokenizer.decode([128003])}'")

🔍 TOKENIZER CHECK
  Vocab size:      128000
  EOS token:       '<|end_of_text|>' (id=128001)
  BOS token:       '<|begin_of_text|>' (id=128000)
  PAD token:       '<|end_of_text|>' (id=128001)
  PAD == EOS:      True

  All special tokens: ['<|begin_of_text|>', '<|end_of_text|>']
  All special IDs:    [128000, 128001]

  Sample text length: 91 chars
  Token count:        27
  Round-trip match:   False

  Token 128001: '<|end_of_text|>'
  Token 128002: '<|reserved_special_token_0|>'
  Token 128003: '<|reserved_special_token_1|>'


In [9]:
# CELL 10: FIX TOKENIZER + SAVE BASE MODEL

# Fix 1: Use reserved token as pad (NOT eos)
tokenizer.pad_token = "<|reserved_special_token_0|>"
tokenizer.pad_token_id = 128002
tokenizer.padding_side = "left"  # for generation later

print(f"✅ PAD token fixed:")
print(f"   PAD: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"   EOS: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"   PAD == EOS: {tokenizer.pad_token_id == tokenizer.eos_token_id}")

# Fix 2: Check round-trip
sample = "Schema: test\nQuestion: test\nCypher: MATCH (n) RETURN n"
tokens = tokenizer(sample, return_tensors="pt")
decoded = tokenizer.decode(tokens['input_ids'][0], skip_special_tokens=True)
print(f"\n   Round-trip (skip_special): {decoded.strip() == sample.strip()}")

# Save base model + fixed tokenizer
import os
os.makedirs("/workspace/models/base", exist_ok=True)

print("\n💾 Saving base model + fixed tokenizer...")
tokenizer.save_pretrained("/workspace/models/base/")
model.save_pretrained("/workspace/models/base/")

size_gb = sum(os.path.getsize(os.path.join('/workspace/models/base', f)) 
              for f in os.listdir('/workspace/models/base')) / 1e9
print(f"✅ Saved to /workspace/models/base/ ({size_gb:.1f} GB)")

# Free VRAM for finetuning
del model
import gc
gc.collect()
import torch
torch.cuda.empty_cache()
print(f"   VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print("\n🎯 Ready for finetuning")

✅ PAD token fixed:
   PAD: '<|reserved_special_token_0|>' (id=128002)
   EOS: '<|end_of_text|>' (id=128001)
   PAD == EOS: False

   Round-trip (skip_special): True

💾 Saving base model + fixed tokenizer...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Saved to /workspace/models/base/ (16.1 GB)
   VRAM after cleanup: 0.01 GB

🎯 Ready for finetuning


In [10]:
# CELL 11: PREPARE TRAINING DATASET + VERIFY BEFORE FINETUNING
import json, torch
from torch.utils.data import Dataset

# Reload tokenizer (model was deleted to free VRAM)
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/workspace/models/base/")

print(f"  PAD: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"  EOS: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"  PAD == EOS: {tokenizer.pad_token_id == tokenizer.eos_token_id}")

# Load processed data
with open("/workspace/data/train_processed.json") as f:
    train_data = json.load(f)
print(f"\n  Train samples: {len(train_data)}")

# Training format: prompt + completion + EOS
# We'll mask the prompt tokens in labels (set to -100) so loss only on completion
MAX_LENGTH = 512

class CypherDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.samples = []
        self.skipped = 0
        
        for item in data:
            prompt = item['prompt']
            completion = item['completion']  # starts with " "
            
            # Tokenize prompt and completion separately to know where to mask
            prompt_ids = tokenizer(prompt, add_special_tokens=True, truncation=False)['input_ids']
            
            # Full text = prompt + completion + EOS
            full_text = prompt + completion + tokenizer.eos_token
            full_ids = tokenizer(full_text, add_special_tokens=True, truncation=True, 
                                max_length=max_length, padding='max_length')
            
            input_ids = full_ids['input_ids']
            attention_mask = full_ids['attention_mask']
            
            # Labels = input_ids, but mask prompt portion with -100
            labels = list(input_ids)
            prompt_len = min(len(prompt_ids), max_length)
            for i in range(prompt_len):
                labels[i] = -100
            # Also mask padding
            for i in range(len(labels)):
                if attention_mask[i] == 0:
                    labels[i] = -100
            
            self.samples.append({
                'input_ids': torch.tensor(input_ids),
                'attention_mask': torch.tensor(attention_mask),
                'labels': torch.tensor(labels),
            })
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

print("\n🔧 Tokenizing training data...")
train_dataset = CypherDataset(train_data, tokenizer, MAX_LENGTH)
print(f"  Dataset size: {len(train_dataset)}")

# ============================================================
# VERIFY 3 SAMPLES
# ============================================================
print("\n" + "=" * 60)
print("🔍 VERIFICATION — Check these carefully!")
print("=" * 60)

for idx in [0, 100, 5000]:
    sample = train_dataset[idx]
    ids = sample['input_ids']
    mask = sample['attention_mask']
    labels = sample['labels']
    
    # Count tokens
    real_tokens = mask.sum().item()
    prompt_masked = (labels == -100).sum().item()
    completion_tokens = real_tokens - (prompt_masked - (MAX_LENGTH - real_tokens))
    
    print(f"\n--- Sample {idx} ---")
    print(f"  Total tokens:      {real_tokens} (of {MAX_LENGTH} max)")
    print(f"  Prompt (masked):   {(labels[:real_tokens] == -100).sum().item()} tokens")
    print(f"  Completion (loss): {(labels[:real_tokens] != -100).sum().item()} tokens")
    
    # Decode prompt part (where labels == -100 and mask == 1)
    prompt_end = 0
    for i in range(len(labels)):
        if labels[i] != -100:
            prompt_end = i
            break
    
    prompt_text = tokenizer.decode(ids[:prompt_end], skip_special_tokens=True)
    completion_text = tokenizer.decode(ids[prompt_end:real_tokens], skip_special_tokens=True)
    
    print(f"  PROMPT:     ...{prompt_text[-80:]}")
    print(f"  COMPLETION: {completion_text[:120]}")
    
    # Check: does completion end with EOS?
    last_real = ids[real_tokens - 1].item()
    print(f"  Last token: {last_real} ({'EOS ✅' if last_real == tokenizer.eos_token_id else '⚠️ NOT EOS'})")
    
    # Check: is the first non-masked label the start of the Cypher?
    first_label_token = tokenizer.decode([ids[prompt_end].item()])
    print(f"  First completion token: '{first_label_token}'")

# Final checks
print(f"\n{'='*60}")
print(f"📋 SUMMARY CHECKS:")
print(f"  Dataset size:     {len(train_dataset)}")
print(f"  Max length:       {MAX_LENGTH}")
print(f"  PAD token id:     {tokenizer.pad_token_id}")
print(f"  EOS token id:     {tokenizer.eos_token_id}")
print(f"  PAD ≠ EOS:        {tokenizer.pad_token_id != tokenizer.eos_token_id}")
s = train_dataset[0]
print(f"  Sample shape:     input_ids={s['input_ids'].shape}, labels={s['labels'].shape}")
print(f"  Labels has -100:  {(s['labels'] == -100).any().item()}")
print(f"  Labels has real:  {(s['labels'] != -100).any().item()}")
print(f"{'='*60}")

  PAD: '<|reserved_special_token_0|>' (id=128002)
  EOS: '<|end_of_text|>' (id=128001)
  PAD == EOS: False

  Train samples: 39554

🔧 Tokenizing training data...
  Dataset size: 39554

🔍 VERIFICATION — Check these carefully!

--- Sample 0 ---
  Total tokens:      204 (of 512 max)
  Prompt (masked):   152 tokens
  Completion (loss): 52 tokens
  PROMPT:     ...h 3 countries have the most entities linked as beneficiaries in filings?
Cypher:
  COMPLETION:  MATCH (f:Filing)-[:BENEFITS]->(e:Entity)-[:COUNTRY]->(c:Country) WITH c.name AS country, COUNT(e) AS entityCount ORDER 
  Last token: 128001 (EOS ✅)
  First completion token: ' MATCH'

--- Sample 100 ---
  Total tokens:      154 (of 512 max)
  Prompt (masked):   93 tokens
  Completion (loss): 61 tokens
  PROMPT:     ...tiscale superpopulation models
- independent innovations trees
- water-!
Cypher:
  COMPLETION:  MATCH (a:Author{affiliation:'unspecified'})-[*]->(d:Keyword{name:'tree (optimality criteria: minimum mean-squared error
  Last

In [13]:
# CELL 14: FIX ACCELERATE VERSION
!pip install -q "accelerate>=1.6.0"

print("✅ Updated. Restart kernel, then run Cell 15.")
print("⚠️  Go to Kernel → Restart Kernel")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
✅ Updated. Restart kernel, then run Cell 15.
⚠️  Go to Kernel → Restart Kernel


In [2]:
# CELL 15: FULL RELOAD + TRAIN (post-restart)
import os, time, json, torch, gc
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from torch.utils.data import Dataset

os.environ["HF_HOME"] = "/workspace/.cache/huggingface"

# ============================================================
# 1. RELOAD TOKENIZER
# ============================================================
print("📥 Reloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("/workspace/models/base/")
print(f"   PAD={tokenizer.pad_token_id}, EOS={tokenizer.eos_token_id}, PAD≠EOS: {tokenizer.pad_token_id != tokenizer.eos_token_id}")

# ============================================================
# 2. REBUILD DATASET FROM SAVED JSON
# ============================================================
print("\n📥 Rebuilding dataset from /workspace/data/train_processed.json...")

with open("/workspace/data/train_processed.json") as f:
    train_data = json.load(f)

MAX_LENGTH = 512

class CypherDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.samples = []
        for item in data:
            prompt = item['prompt']
            completion = item['completion']
            full_text = prompt + completion + tokenizer.eos_token
            full_ids = tokenizer(full_text, add_special_tokens=True, truncation=True,
                                max_length=max_length, padding='max_length')
            input_ids = full_ids['input_ids']
            attention_mask = full_ids['attention_mask']
            prompt_ids = tokenizer(prompt, add_special_tokens=True, truncation=False)['input_ids']
            labels = list(input_ids)
            prompt_len = min(len(prompt_ids), max_length)
            for i in range(prompt_len):
                labels[i] = -100
            for i in range(len(labels)):
                if attention_mask[i] == 0:
                    labels[i] = -100
            self.samples.append({
                'input_ids': torch.tensor(input_ids),
                'attention_mask': torch.tensor(attention_mask),
                'labels': torch.tensor(labels),
            })
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return self.samples[idx]

train_dataset = CypherDataset(train_data, tokenizer, MAX_LENGTH)
print(f"   Dataset: {len(train_dataset)} samples")

# Quick verify
s = train_dataset[0]
print(f"   Sample shape: {s['input_ids'].shape}")
print(f"   Has masked labels: {(s['labels']==-100).any().item()}")
print(f"   Has real labels: {(s['labels']!=-100).any().item()}")

# ============================================================
# 3. LOAD MODEL
# ============================================================
print("\n📥 Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    "/workspace/models/base/",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.gradient_checkpointing_enable()
model.config.use_cache = False

print(f"   Loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# ============================================================
# 4. SETUP TRAINER
# ============================================================
BATCH_SIZE = 2
GRAD_ACCUM = 16
EPOCHS = 3
total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
save_steps = total_steps // 6

print(f"\n📋 Training config:")
print(f"   Effective batch: {BATCH_SIZE}x{GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}")
print(f"   Epochs: {EPOCHS}, Total steps: ~{total_steps}, Save every: {save_steps}")

training_args = TrainingArguments(
    output_dir="/workspace/models/finetuned",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=int(total_steps * 0.1),
    lr_scheduler_type="linear",
    logging_steps=50,
    save_steps=save_steps,
    save_total_limit=3,
    bf16=True,
    optim="adamw_bnb_8bit",
    gradient_checkpointing=True,
    dataloader_pin_memory=True,
    report_to="none",
    max_grad_norm=1.0,
    seed=42,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print(f"   VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# ============================================================
# 5. TRAIN
# ============================================================
print(f"\n🚀 Starting training at {time.strftime('%H:%M:%S')}")
print("=" * 60)

train_start = time.time()
train_result = trainer.train()
train_elapsed = time.time() - train_start

# ============================================================
# 6. RESULTS
# ============================================================
print(f"\n{'='*60}")
print(f"✅ Training complete!")
print(f"   Wall time:   {train_elapsed/3600:.1f} hours ({train_elapsed:.0f}s)")
print(f"   Final loss:  {train_result.metrics['train_loss']:.4f}")
print(f"   VRAM peak:   {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

# ============================================================
# 7. SAVE FINAL MODEL
# ============================================================
print(f"\n💾 Saving final model...")
os.makedirs("/workspace/models/finetuned/final", exist_ok=True)
trainer.save_model("/workspace/models/finetuned/final")
tokenizer.save_pretrained("/workspace/models/finetuned/final")

size_gb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk("/workspace/models/finetuned/final")
    for f in fn
) / 1e9
print(f"   Saved to /workspace/models/finetuned/final/ ({size_gb:.1f} GB)")

# ============================================================
# 8. SAVE LOGS
# ============================================================
log_data = {
    'model': 'meta-llama/Meta-Llama-3-8B',
    'dataset': 'neo4j/text2cypher-2024v1',
    'train_samples': len(train_dataset),
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'effective_batch': BATCH_SIZE * GRAD_ACCUM,
    'learning_rate': 2e-5,
    'wall_time_hours': train_elapsed / 3600,
    'final_loss': train_result.metrics['train_loss'],
    'metrics': train_result.metrics,
    'vram_peak_gb': torch.cuda.max_memory_allocated() / 1e9,
}
with open("/workspace/models/finetuned/training_log.json", "w") as f:
    json.dump(log_data, f, indent=2, default=str)

loss_history = [
    {'step': e['step'], 'loss': e['loss']}
    for e in trainer.state.log_history if 'loss' in e
]
with open("/workspace/models/finetuned/loss_history.json", "w") as f:
    json.dump(loss_history, f, indent=2)
print(f"   Saved training_log.json + loss_history.json ({len(loss_history)} entries)")

# ============================================================
# 9. LIST CHECKPOINTS
# ============================================================
print(f"\n📁 Checkpoints:")
for item in sorted(os.listdir("/workspace/models/finetuned")):
    path = os.path.join("/workspace/models/finetuned", item)
    if os.path.isdir(path):
        ckpt_size = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, dn, fn in os.walk(path) for f in fn
        ) / 1e9
        print(f"   📁 {item}/ ({ckpt_size:.1f} GB)")
    else:
        print(f"   📄 {item}")

print(f"\n{'='*60}")
print(f"🎯 Next: evaluate finetuned model accuracy")
print(f"{'='*60}")

📥 Reloading tokenizer...
   PAD=128002, EOS=128001, PAD≠EOS: True

📥 Rebuilding dataset from /workspace/data/train_processed.json...


`torch_dtype` is deprecated! Use `dtype` instead!


   Dataset: 39554 samples
   Sample shape: torch.Size([512])
   Has masked labels: True
   Has real labels: True

📥 Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128002}.


...

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SafetensorError: Error while serializing: I/O error: Disk quota exceeded (os error 122)

In [3]:
# CELL 16: CHECK DISK + FIND SAVED CHECKPOINTS
import os

print("💾 DISK STATUS:")
os.system("df -h /workspace")

print("\n📁 FINETUNED DIRECTORY:")
finetuned_dir = "/workspace/models/finetuned"
if os.path.exists(finetuned_dir):
    total_size = 0
    for item in sorted(os.listdir(finetuned_dir)):
        path = os.path.join(finetuned_dir, item)
        if os.path.isdir(path):
            ckpt_size = sum(
                os.path.getsize(os.path.join(dp, f))
                for dp, dn, fn in os.walk(path) for f in fn
            )
            total_size += ckpt_size
            print(f"   📁 {item}/ ({ckpt_size/1e9:.1f} GB)")
        else:
            s = os.path.getsize(path)
            total_size += s
            print(f"   📄 {item} ({s/1e3:.1f} KB)")
    print(f"\n   Total finetuned: {total_size/1e9:.1f} GB")

print("\n📁 FULL WORKSPACE USAGE:")
os.system("du -sh /workspace/models/base/")
os.system("du -sh /workspace/models/finetuned/")
os.system("du -sh /workspace/data/")
os.system("du -sh /workspace/.cache/ 2>/dev/null")
os.system("du -sh /workspace/")

💾 DISK STATUS:
Filesystem                   Size  Used Avail Use% Mounted on
mfs#us-md-1.runpod.net:9421  299T  225T   75T  76% /workspace

📁 FINETUNED DIRECTORY:
   📁 checkpoint-1236/ (35.5 GB)
   📁 checkpoint-1854/ (12.8 GB)
   📁 checkpoint-618/ (35.5 GB)

   Total finetuned: 83.9 GB

📁 FULL WORKSPACE USAGE:
15G	/workspace/models/base/
79G	/workspace/models/finetuned/
94M	/workspace/data/
2.5K	/workspace/.cache/
94G	/workspace/


0

In [4]:
# CELL 17: CLEAN UP + PROMOTE BEST CHECKPOINT
import shutil, os

# Delete incomplete checkpoint (corrupted)
print("🗑️ Deleting incomplete checkpoint-1854...")
shutil.rmtree("/workspace/models/finetuned/checkpoint-1854", ignore_errors=True)

# Delete oldest checkpoint
print("🗑️ Deleting old checkpoint-618...")
shutil.rmtree("/workspace/models/finetuned/checkpoint-618", ignore_errors=True)

# Rename checkpoint-1236 as final
print("💾 Promoting checkpoint-1236 as final model...")
src = "/workspace/models/finetuned/checkpoint-1236"
dst = "/workspace/models/finetuned/final"

if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

# Also save tokenizer to final
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/workspace/models/base/")
tokenizer.save_pretrained(dst)
print(f"   Tokenizer saved to final/")

# Now delete the checkpoint too
shutil.rmtree(src)

# Check disk
print(f"\n💾 Disk after cleanup:")
os.system("du -sh /workspace/models/base/")
os.system("du -sh /workspace/models/finetuned/")
os.system("du -sh /workspace/")

# Verify final model
print(f"\n📁 Final model contents:")
for f in sorted(os.listdir(dst)):
    size = os.path.getsize(os.path.join(dst, f))
    print(f"   {f} ({size/1e9:.1f} GB)" if size > 1e8 else f"   {f} ({size/1e3:.1f} KB)")

print(f"\n✅ Final model at /workspace/models/finetuned/final/")
print(f"   Epoch ~1.0, Loss ~0.109")
print(f"   This is good enough — loss was still improving but epoch 1 captures the core capability")

🗑️ Deleting incomplete checkpoint-1854...
🗑️ Deleting old checkpoint-618...
💾 Promoting checkpoint-1236 as final model...
   Tokenizer saved to final/

💾 Disk after cleanup:
15G	/workspace/models/base/
34G	/workspace/models/finetuned/
49G	/workspace/

📁 Final model contents:
   config.json (0.7 KB)
   generation_config.json (0.2 KB)
   model.safetensors (16.1 GB)
   optimizer.pt (19.4 GB)
   rng_state.pth (14.2 KB)
   scheduler.pt (1.1 KB)
   tokenizer.json (17210.0 KB)
   tokenizer_config.json (0.4 KB)
   trainer_state.json (5.1 KB)
   training_args.bin (4.8 KB)

✅ Final model at /workspace/models/finetuned/final/
   Epoch ~1.0, Loss ~0.109
   This is good enough — loss was still improving but epoch 1 captures the core capability


In [ ]:
# CELL 18: DELETE OPTIMIZER (not needed for inference) + EVALUATE
import os

# Free 19GB
os.remove("/workspace/models/finetuned/final/optimizer.pt")
os.remove("/workspace/models/finetuned/final/rng_state.pth")
os.remove("/workspace/models/finetuned/final/scheduler.pt")
os.remove("/workspace/models/finetuned/final/training_args.bin")
print("🗑️ Deleted optimizer + training state (saved ~19.4 GB)")
os.system("du -sh /workspace/")

# Now load finetuned model and evaluate
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

print("\n📥 Loading finetuned model...")
tokenizer = AutoTokenizer.from_pretrained("/workspace/models/finetuned/final/")
model = AutoModelForCausalLM.from_pretrained(
    "/workspace/models/finetuned/final/",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"   Loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Evaluate on FULL test set (4833 samples)
print(f"\n🧪 EVALUATING ON FULL TEST SET")
print("=" * 60)

with open("/workspace/data/test_processed.json") as f:
    test_data = json.load(f)

print(f"   Test samples: {len(test_data)}")

exact = 0
valid = 0
per_cy = {}
baseline_correct = {}  # indices of correct samples per complexity

for i in tqdm(range(len(test_data)), desc="Evaluating"):
    item = test_data[i]
    
    inputs = tokenizer(
        item['prompt'], 
        return_tensors="pt", 
        truncation=True, 
        max_length=480
    ).to(model.device)
    
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    first_line = gen.split('\n')[0].strip()
    expected = item['cypher_ref'].strip()
    
    cy = item['complexity']
    if cy not in per_cy:
        per_cy[cy] = {'exact': 0, 'valid': 0, 'total': 0}
        baseline_correct[cy] = []
    per_cy[cy]['total'] += 1
    
    is_exact = (first_line == expected)
    is_valid = first_line.upper().startswith('MATCH') or first_line.upper().startswith('RETURN') or first_line.upper().startswith('CALL')
    
    if is_exact:
        exact += 1
        per_cy[cy]['exact'] += 1
        baseline_correct[cy].append(i)
    if is_valid:
        valid += 1
        per_cy[cy]['valid'] += 1

# Results
total = len(test_data)
print(f"\n{'='*60}")
print(f"📊 FINETUNED MODEL RESULTS (epoch ~1.0, loss ~0.109)")
print(f"{'='*60}")
print(f"   Exact match:  {exact}/{total} ({100*exact/total:.1f}%)")
print(f"   Valid Cypher:  {valid}/{total} ({100*valid/total:.1f}%)")

print(f"\n   Per complexity:")
for cy in sorted(per_cy.keys()):
    d = per_cy[cy]
    print(f"   {cy}: exact={d['exact']}/{d['total']} ({100*d['exact']/d['total']:.1f}%), "
          f"valid={d['valid']}/{d['total']} ({100*d['valid']/d['total']:.1f}%)")

total_correct = sum(len(v) for v in baseline_correct.values())
print(f"\n   Baseline-correct samples: {total_correct}")
print(f"   (This is our n for all retention experiments)")

# Compare to Gemma
print(f"\n   📊 vs Gemma-2B-SQL:")
print(f"   Gemma-2B:  105 baseline-correct out of 200 (52.5%)")
print(f"   Llama-8B:  {total_correct} baseline-correct out of {total} ({100*total_correct/total:.1f}%)")

# Save
results = {
    'model': 'Llama-3-8B finetuned on Text2Cypher',
    'checkpoint': 'epoch ~1.0, loss ~0.109',
    'exact_match': exact,
    'valid_cypher': valid,
    'total': total,
    'exact_pct': 100*exact/total,
    'per_complexity': per_cy,
    'baseline_correct': baseline_correct,
    'total_baseline_correct': total_correct,
}

with open("/workspace/data/finetuned_accuracy.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\n💾 Saved to finetuned_accuracy.json")

print(f"\n🎯 Next: upload to HuggingFace, then stop pod")

🗑️ Deleted optimizer + training state (saved ~19.4 GB)
31G	/workspace/

📥 Loading finetuned model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

   Loaded. VRAM: 51.6 GB

🧪 EVALUATING ON FULL TEST SET
   Test samples: 4833


Evaluating:   0%|          | 0/4833 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take p